In [18]:
import pandas as pd
import psycopg2
import pandas as pd
import geopandas as gpd
import requests
from geopandas.tools import sjoin
import time
import datetime


In [19]:
df_patients = pd.read_csv("H:/canc_air/data/data_octobre_2023/Pseudonymisation_provisoire.csv", sep=";",encoding_errors='ignore')

In [20]:
def patient_to_geocoding(df):
    df["requete"] =df['adresse'] + " " + df["codepost"]+ " " + df["nom_commune_postal"]
    return df 

df_patients_ready = patient_to_geocoding(df_patients)

In [21]:
def geocode(df):
    """Geocode dataframe with Etalab addok"""
    print("Proceed to geocode on address...")
    for i in df.index:
        try:
            # get json response
            r = requests.get('https://addok-data.curie.net/search?q='+df["requete"][i])
            response = r.json()

            if i%100==0 : 
                print(f"Proceed geocoding at the {i}th row")
            if response["features"]!=[]:
                # parse json to insert value in dataframe
                #df.at[i, 'nip'] = str(df["pseudo_provisoire"][i])
                df.at[i, 'x'] = str(response["features"][0]["geometry"]["coordinates"][0])
                df.at[i, 'y'] = str(response["features"][0]["geometry"]["coordinates"][1])
                df.at[i, 'score'] = str(response["features"][0]["properties"]["score"])

                if float(df["score"][i])<0.4:
                    df.at[i, 'trust_score'] = 'low'
                elif float(df["score"][i])>0.4 and float(df["score"][i])<0.65:
                    df.at[i, 'trust_score'] = 'middle'
                elif float(df["score"][i])>0.65 and float(df["score"][i])<0.9:
                    df.at[i, 'trust_score'] = 'middle'
                else:
                    df.at[i, 'trust_score'] = 'high'

                df.at[i, 'street'] = str(response["features"][0]["properties"]["name"]).replace("'", " ")
                df.at[i, 'city'] = str(response["features"][0]["properties"]["city"]).replace("'", " ")
                df.at[i, 'pc_city'] = str(response["features"][0]["properties"]["postcode"])
                df.at[i, 'ic_city'] = str(response["features"][0]["properties"]["citycode"])

                context = (str(response["features"][0]["properties"]["context"]).replace("'", " ")).split(",")
                df.at[i, 'code_dept'] = context[0]
                df.at[i, 'dept'] = context[1]

                if len(df["code_dept"][i])==2:
                    df.at[i, 'reg'] = context[2]
                else:
                    df.at[i, 'reg'] = "other"
                #df.at[i, 'code_country'] = str(df["pays"][i])
                df.at[i, 'address'] = str(response["features"][0]["properties"]["label"]).replace("'", " ")
                
                if (df["requete"][i].lstrip())[0].isdigit():
                    df.at[i, 'address_has_num_init'] = "true"
                else:
                    df.at[i, 'address_has_num_init'] = "false"
                    
                if (df["street"][i].lstrip())[0].isdigit():
                    df.at[i, 'address_has_num_geoloc'] = "true"
                else:
                    df.at[i, 'address_has_num_geoloc'] = "false"
                    
                if df["codepost"][i] == df["pc_city"][i]:
                    df.at[i, 'same_city'] = "true"
                else:
                    df.at[i, 'same_city'] = "false" 
                    
                if "hotel" in df["street"][i] or "hôtel" in df["street"][i]: 
                    df.at[i, 'hostel'] = "true"
                else:
                    df.at[i, 'hostel'] = "false"

                if "chez" in df["street"][i]: 
                    df.at[i, 'hosted'] = "true"
                else:
                    df.at[i, 'hosted'] = "false"
                    
                df.at[i, 'date_geoloc'] = str(datetime.date.today())
                #df.at[i, 'etalab_version'] = str(response["licence"])
                #df.at[i, 'ban_version'] = "2021-04-27"
            else:
                pass
        except:
            pass
    return df
    
def spatialjoin(df, df_iris, df_region, df_dept):
    """Spatial join of the database and the iris/dept/region layers"""
    print("Perfom to spatial join on IRIS and EPCI layers...")
    gdf = (gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y)).set_crs(epsg=4326)).to_crs(epsg=2154)

    df_join_iris = gpd.sjoin(gdf, df_iris[['CODE_IRIS','geometry']], how="left", op='within')
    df_join_iris.drop('index_right', axis=1, inplace=True)

    df_region =df_region.to_crs(epsg=2154)
    df_join_iris_reg = gpd.sjoin(df_join_iris, df_region[['INSEE_REG','geometry']], how="left", op='within')
    df_join_iris_reg.drop('index_right', axis=1, inplace=True)

    df_join_iris_reg_dept = gpd.sjoin(df_join_iris_reg, df_dept[['CODE_DEPT','geometry']], how="left", op='within')
    df_join_iris_reg_dept.drop('index_right', axis=1, inplace=True)
    return df_join_iris_reg_dept



In [22]:
df_geocoded = geocode(df_patients_ready)
df_geocoded.to_csv("H:/canc_air/data/data_cleaned/Pseudonymisation_provisoire_geocoded.csv", sep=";")
#df_geocoded = pd.read_csv("H:/canc_air/data/data_cleaned/Pseudonymisation_provisoire_geocoded.csv", sep=";")
#df_geocoded.drop(['Unnamed: 0.1','Unnamed: 0'], axis=1, inplace=True)
#df_geocoded.head()

Proceed to geocode on address...
Proceed geocoding at the 0th row
Proceed geocoding at the 100th row
Proceed geocoding at the 200th row
Proceed geocoding at the 300th row
Proceed geocoding at the 400th row
Proceed geocoding at the 500th row
Proceed geocoding at the 600th row
Proceed geocoding at the 700th row
Proceed geocoding at the 800th row
Proceed geocoding at the 900th row
Proceed geocoding at the 1000th row
Proceed geocoding at the 1100th row
Proceed geocoding at the 1200th row
Proceed geocoding at the 1300th row
Proceed geocoding at the 1400th row
Proceed geocoding at the 1500th row
Proceed geocoding at the 1600th row
Proceed geocoding at the 1700th row
Proceed geocoding at the 1800th row
Proceed geocoding at the 1900th row
Proceed geocoding at the 2000th row
Proceed geocoding at the 2100th row
Proceed geocoding at the 2200th row
Proceed geocoding at the 2300th row
Proceed geocoding at the 2400th row
Proceed geocoding at the 2500th row
Proceed geocoding at the 2600th row
Proceed

In [5]:
df_iris = gpd.read_file('H:/canc_air/data/zones_geographiques/iris/CONTOURS-IRIS.shp')
df_region = gpd.read_file("H:/canc_air/data/zones_geographiques/region/REGION.shp")
df_dept = gpd.read_file('H:/canc_air/data/zones_geographiques/departements/DEPARTEMENT.shp')

In [23]:
df_geocoded_spatial= spatialjoin(df_geocoded, df_iris, df_region, df_dept)
df_geocoded_spatial.to_csv("H:/canc_air/data/data_cleaned/Pseudonymisation_provisoire_geocoded_dpt_reg.csv", sep=";")

Perfom to spatial join on IRIS and EPCI layers...


C:\Users\jbocque1\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3548: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
C:\Users\jbocque1\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3548: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
C:\Users\jbocque1\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3548: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [14]:
print(f"Le nombre de patients totaux initial : {len(df_geocoded_spatial)}")
print(f"Le nombre de patients avec une adresse : {len(df_geocoded_spatial)-df_geocoded_spatial['adresse'].isnull().sum()}. verification : {len(df_geocoded_spatial_wout_na_adresse)}")
print(f"Le nombre de patients sans adresses : {df_geocoded_spatial['adresse'].isnull().sum()}")
print(f"Le nombre de patient en France Métropolitaine avec une adresse : {len(patients_in_france)}")


Le nombre de patients totaux initial : 64294
Le nombre de patients avec une adresse : 62638. verification : 62638
Le nombre de patients sans adresses : 1656
Le nombre de patient en France Métropolitaine avec une adresse : 59676


In [57]:
print(f"Le nombre de patients totaux : {len(df_geocoded_spatial)}")
print(f"Le nombre de patients avec une adresse : {len(df_geocoded_spatial)-df_geocoded_spatial['adresse'].isnull().sum()}. verification : {len(df_geocoded_spatial_wout_na_adresse)}")
print(f"Le nombre de patients présent en france métropolitaine : {len(patients_in_france)}")
print(f"Le nombre de patients comprennant un EPCI, Iris et ")

Le nombre de patients totaux : 64294
Le nombre de patients avec une adresse : 62638. verification : 62638
Le nombre de patients présent en france métropolitaine : 59676
